# Predicting House Prices

In [1]:
# The data is available on AWS S3 at https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv

In [13]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn import datasets
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error
from google.colab import userdata
import os


In [3]:
base_data = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv"

In [4]:
housing_p = pd.read_csv(base_data)
housing_p

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,903231070,50,RM,52.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,400,9,2006,WD,Normal,114500
2633,906201021,80,RL,74.0,10778,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,WD,Normal,162000
2634,533253070,120,RL,61.0,3782,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2009,WD,Normal,211500
2635,527376100,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,8,2009,WD,Normal,165000


## Cleaning

In [5]:
housing_p_c = housing_p.copy()

In [6]:
# getting rid of the rows that have nulls
housing_p_c.drop(columns = ["PID", 'Fence', 'Misc Feature', 'Alley', 'Pool QC', 'Mas Vnr Type', 'Fireplace Qu', 'Lot Frontage', 'Garage Yr Blt', 'Garage Finish', 'Garage Qual', 'Garage Cond', 'Garage Type', 'Bsmt Exposure', 'BsmtFin Type 2'],inplace = True)

In [7]:
housing_p_c.drop(columns = ['Bsmt Qual', 'BsmtFin Type 1', 'Bsmt Cond', 'Mas Vnr Area', 'Bsmt Half Bath', 'Bsmt Full Bath', 'Garage Cars', 'Garage Area',],inplace = True)

In [8]:
housing_p_c.drop(columns = ['BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF'], inplace = True)

In [9]:
housing_p_c = housing_p_c.select_dtypes(exclude=['object'])

In [10]:
#dropped row from 81 to 26
housing_p_c.shape

(2637, 26)

In [11]:
housing_p_c.isnull().sum().sort_values( ascending = False ).head(10)

,0
MS SubClass,0
Lot Area,0
Overall Qual,0
Overall Cond,0
Year Built,0
Year Remod/Add,0
1st Flr SF,0
2nd Flr SF,0
Low Qual Fin SF,0
Gr Liv Area,0


In [12]:
housing_p_c.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   MS SubClass      2637 non-null   int64
 1   Lot Area         2637 non-null   int64
 2   Overall Qual     2637 non-null   int64
 3   Overall Cond     2637 non-null   int64
 4   Year Built       2637 non-null   int64
 5   Year Remod/Add   2637 non-null   int64
 6   1st Flr SF       2637 non-null   int64
 7   2nd Flr SF       2637 non-null   int64
 8   Low Qual Fin SF  2637 non-null   int64
 9   Gr Liv Area      2637 non-null   int64
 10  Full Bath        2637 non-null   int64
 11  Half Bath        2637 non-null   int64
 12  Bedroom AbvGr    2637 non-null   int64
 13  Kitchen AbvGr    2637 non-null   int64
 14  TotRms AbvGrd    2637 non-null   int64
 15  Fireplaces       2637 non-null   int64
 16  Wood Deck SF     2637 non-null   int64
 17  Open Porch SF    2637 non-null   int64
 18  Enclosed

## Convert CSV to parquet


In [14]:
parquet_file = "housing_p_c.parquet"
parquet_file


'housing_p_c.parquet'

In [15]:
housing_p_c.to_parquet( parquet_file, index=False)

In [16]:
!ls -l --si {parquet_file}


-rw-r--r-- 1 root root 86k Jul  1 14:36 housing_p_c.parquet


## Verify that pandas can query the local parquet file


In [17]:
df2 = pd.read_parquet( parquet_file )
df2.shape


(2637, 26)

## Authenticate to Hugging Face


In [18]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_nW ... OZU'

In [20]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account


'zenithvalor'

In [24]:
hf_repo = "data_science_21"
os.environ["HF_REPO"] = hf_repo
hf_repo


'data_science_21'

In [22]:
# Log in automatically without an interactive prompt
!hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (1.21.0) is available! You are using version 1.20.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Push to Hugging Face


In [25]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  housing_p_c.parquet


In [26]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/zenithvalor/data_science_21/commit/e47319539ca1f0f9fc6e0a15dd9a6fd05e2cd8dc



## Read parquet from Hugging Face


In [27]:
# Construct the base Hugging Face URL for your dataset files
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/housing_p_c.parquet"
hf_url


'https://huggingface.co/datasets/zenithvalor/data_science_21/resolve/main/housing_p_c.parquet'

In [28]:
df3 = pd.read_parquet( hf_url )
df3.shape


(2637, 26)

In [29]:
df3.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   MS SubClass   2637 non-null   int64
 1   Lot Area      2637 non-null   int64
 2   Overall Qual  2637 non-null   int64
 3   Overall Cond  2637 non-null   int64
 4   Year Built    2637 non-null   int64
dtypes: int64(5)
memory usage: 103.1 KB


In [30]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   MS SubClass      2637 non-null   int64
 1   Lot Area         2637 non-null   int64
 2   Overall Qual     2637 non-null   int64
 3   Overall Cond     2637 non-null   int64
 4   Year Built       2637 non-null   int64
 5   Year Remod/Add   2637 non-null   int64
 6   1st Flr SF       2637 non-null   int64
 7   2nd Flr SF       2637 non-null   int64
 8   Low Qual Fin SF  2637 non-null   int64
 9   Gr Liv Area      2637 non-null   int64
 10  Full Bath        2637 non-null   int64
 11  Half Bath        2637 non-null   int64
 12  Bedroom AbvGr    2637 non-null   int64
 13  Kitchen AbvGr    2637 non-null   int64
 14  TotRms AbvGrd    2637 non-null   int64
 15  Fireplaces       2637 non-null   int64
 16  Wood Deck SF     2637 non-null   int64
 17  Open Porch SF    2637 non-null   int64
 18  Enclosed